# Семинар 08. Итераторы и генераторы

На семинаре соберём ленивый конвейер обработки текстового отчёта, разобьём поток на пакеты и объединим два отсортированных журнала без загрузки всех записей в память.

## Цели

- вручную пройти протокол `iter` / `next`;
- писать генераторные функции и выражения;
- сохранять ленивость между этапами конвейера;
- использовать `yield from` и `itertools`;
- работать с одноразовыми входами без `len()` и индексов;
- тестировать раннее завершение и момент возникновения ошибок.

## Перед началом

Откройте [лекцию](lecture.ipynb). В каждом задании сначала решите, что функция должна принимать — `Iterable` или конкретный список — и что возвращать. Не оборачивайте вход в `list()` ради удобства: одноразовый генератор сразу обнаружит такую подмену потокового решения.

## Задание 1. Два независимых обхода

Создайте два итератора одного списка операций. Первым извлеките две записи, вторым — одну. Проверьте позиции через следующие `next()`. Затем вручную реализуйте аналог `for`, собирающий оставшиеся `id` первого итератора до `StopIteration`.

In [ ]:
operations = [
    {"id": "o-1"},
    {"id": "o-2"},
    {"id": "o-3"},
    {"id": "o-4"},
]

first = iter(operations)
second = iter(operations)

# TODO: сделайте указанные шаги через next().
# TODO: соберите остаток first циклом while и try/except.

## Задание 2. Доказываем ленивость

Допишите генератор завершённых операций. В список `visited` записывайте `id` каждой просмотренной записи. После создания генератора список должен быть пуст; после первого `next()` — содержать только фактически просмотренные записи.

In [ ]:
visited = []

def iter_completed(operations):
    for operation in operations:
        # TODO: зарегистрируйте просмотр и выдайте только completed.
        pass

source = [
    {"id": "o-1", "status": "pending"},
    {"id": "o-2", "status": "completed"},
    {"id": "o-3", "status": "completed"},
]
stream = iter_completed(source)
# TODO: проверки до и после next(stream).

## Задание 3. Значимые строки

Напишите генератор, который удаляет пробелы по краям и пропускает пустые строки и комментарии. Проверьте его и на списке, и на `StringIO`: функция не должна зависеть от `len()` или индексов.

In [ ]:
from io import StringIO

def meaningful_lines(lines):
    # TODO: выдавайте очищенные значимые строки.
    pass

list_source = ["  t-1;food;900 ", "", " # comment", "t-2;rent;45000"]
file_source = StringIO("# report\nt-3;taxi;300\n\n")
# TODO: проверки для обоих источников.

## Задание 4. Разбор одной строки и всего потока

Сначала напишите обычную функцию `parse_transaction(line) -> dict`. Затем генератор `parse_transactions(lines)`, который применяет её к каждому элементу. Почему разбор одной записи лучше оставить отдельной обычной функцией?

In [ ]:
def parse_transaction(line):
    # TODO: ожидаются id;category;amount и неотрицательная целая сумма.
    pass

def parse_transactions(lines):
    # TODO: применяйте parse_transaction лениво.
    pass

# TODO: обычный случай и несколько некорректных строк.

## Задание 5. Номер ошибки относится к исходному файлу

Соберите `iter_transactions(lines, min_amount)`. Пустые строки и комментарии пропускаются, но номер в `ValueError` должен учитывать их. Используйте `enumerate(lines, start=1)` до фильтрации. Проверьте, что ошибка не возникает при создании генератора, а появляется только при обходе проблемной строки.

In [ ]:
def iter_transactions(lines, min_amount=0):
    # TODO: очистка, разбор, фильтрация и информативный ValueError.
    pass

source = iter([
    "# report",
    "t-1;food;900",
    "",
    "broken line",
])
stream = iter_transactions(source, 500)
# TODO: первый next успешен, второй приводит к ошибке с номером 4.

## Задание 6. Пакеты для отправки

Реализуйте `iter_batches(items, size)`. Вход может быть бесконечным, поэтому нельзя определять его длину или материализовывать целиком. После заполнения пакета отдайте его через `yield` и создайте новый список.

In [ ]:
def iter_batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    # TODO: собирайте и выдавайте пакеты.
    pass

assert list(iter_batches(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]]

## Задание 7. Страницы API через `yield from`

Функция `fetch_page(number)` ниже возвращает `(items, has_next)`. Напишите генератор, который запрашивает страницы по одной и передаёт элементы наружу через `yield from`. После раннего прекращения обхода лишние страницы не должны запрашиваться.

In [ ]:
requested_pages = []

def fetch_page(number):
    requested_pages.append(number)
    pages = {
        1: (["p-1", "p-2"], True),
        2: (["p-3", "p-4"], True),
        3: (["p-5"], False),
    }
    return pages[number]

def iter_api_items(fetch_page):
    # TODO: начинайте с 1, запрашивайте следующую страницу только при необходимости.
    pass

stream = iter_api_items(fetch_page)
# TODO: извлеките три элемента и проверьте requested_pages == [1, 2].

## Задание 8. Собираем инструменты `itertools`

Без ручных циклов:

1. объедините вчерашний и сегодняшний журналы через `chain`;
2. возьмите первые четыре записи через `islice`;
3. создайте номера `REQ-1000`, `REQ-1001`, ... через `count`;
4. сопоставьте четырём событиям номера через `zip(..., strict=True)`.

In [ ]:
from itertools import chain, count, islice

yesterday = ["event-1", "event-2"]
today = ["event-3", "event-4", "event-5"]

# TODO: combined, first_four, request_ids и пары (id, event).

## Задание 9. Слияние двух журналов

Реализуйте ленивое слияние двух источников, уже отсортированных по `timestamp`. Разрешено хранить только текущий элемент каждого источника. Для отсутствующего элемента удобно использовать уникальный sentinel, чтобы не путать его с настоящим `None`.

In [ ]:
def merge_events(left, right):
    left_iterator = iter(left)
    right_iterator = iter(right)
    missing = object()
    left_event = next(left_iterator, missing)
    right_event = next(right_iterator, missing)
    # TODO: выдавайте меньший timestamp; при равенстве сначала left.
    pass

bank = iter([{"id": "b-1", "timestamp": 10}, {"id": "b-2", "timestamp": 20}])
api = iter([{"id": "a-1", "timestamp": 20}, {"id": "a-2", "timestamp": 30}])

## Задание 10. Тестируем потоковые свойства

Создайте `counted_source(values, counter)`, увеличивающий `counter["visited"]` перед каждым `yield`. С его помощью проверьте:

- создание конвейера не читает вход;
- `next()` читает только необходимую часть;
- второй полный обход того же генератора пуст;
- `islice(stream, 2)` не потребляет весь источник;
- некорректная запись вызывает ошибку только при достижении этой записи.

In [ ]:
def counted_source(values, counter):
    for value in values:
        # TODO: увеличьте счётчик перед yield.
        pass

def check_laziness():
    # TODO: проверки перечисленных свойств.
    pass

# check_laziness()

## Обсуждение

Для каждого случая выберите список или генератор и объясните решение:

1. Один раз посчитать сумму большого CSV.
2. Несколько раз строить разные отчёты по одним 500 строкам.
3. Найти первые пять ошибок в бесконечном потоке событий.
4. Отсортировать все транзакции по сумме.
5. Передать API записи пакетами по 100.
6. Вернуть из функции данные файла, который уже закрыт.

В четвёртом случае всё равно потребуется материализация: универсальная сортировка должна видеть весь конечный набор.

## Самопроверка

- Можете объяснить разницу между `Iterable` и `Iterator`?
- Можете доказать тестом, что генератор ленивый?
- Можете обработать вход без `len()` и индексов?
- Можете сохранить номер исходной строки после фильтрации?
- Можете использовать `yield from` для страниц API?
- Можете слить два отсортированных одноразовых потока с `O(1)` памяти?
- Можете назвать операцию, которая незаметно расходует итератор?

## Итоги

На семинаре мы прошли протокол итерации вручную, написали ленивые фильтры и преобразования, собрали потоковый разбор отчёта, разбили данные на пакеты, прошли страницы API и слили два журнала. Отдельные тесты проверяют не только значения, но и важный контракт: сколько входа было реально потреблено и когда возникает ошибка.